# LeetCode 2305 — Fair Distribution of Cookies bằng Python

Link đề: https://leetcode.com/problems/fair-distribution-of-cookies/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Backtracking, Bit Manipulation, Bitmask

Skeleton LeetCode:

```python
class Solution:
    def distributeCookies(self, cookies: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận dạng bài **phân hoạch cân bằng** (minimize makespan / chia việc cho máy) — NP-khó tổng quát, nhưng `n ≤ 8` cho phép vét cạn có cắt tỉa hoặc DP bitmask.
2. Viết **quay lui** (backtracking) với ba kỹ thuật cắt tỉa: sắp xếp giảm dần, cắt theo đáp án tốt nhất hiện có, và **phá đối xứng** (các em đang có cùng số bánh là như nhau).
3. Viết **DP trên tập con**: `f[t][mask]` = độ bất công nhỏ nhất khi chia các túi trong `mask` cho `t` em, duyệt **tập con của tập con** trong O(3ⁿ).
4. Kết hợp **tìm nhị phân đáp án** với kiểm tra khả thi.
5. Thấy vì sao tham lam "túi lớn nhất cho em ít nhất" (LPT) **không** tối ưu bằng phản ví dụ tìm tự động.

## 2. Đề bài & ví dụ

Cho mảng `cookies`, `cookies[i]` là số bánh trong túi thứ `i`, và số trẻ `k`. Chia **mọi** túi cho `k` trẻ; một túi phải trao nguyên cho một trẻ. **Độ bất công** của một cách chia là tổng bánh **lớn nhất** mà một trẻ nhận. Trả về độ bất công **nhỏ nhất** có thể.

**Ví dụ 1:** `cookies = [8,15,10,20,8]`, `k = 2` → `31` (`[8,15,8]` và `[10,20]`).

**Ví dụ 2:** `cookies = [6,1,3,2,2,4,1,2]`, `k = 3` → `7` (`[6,1]`, `[3,2,2]`, `[4,1,2]`).

**Ràng buộc:** `2 ≤ n ≤ 8`; `1 ≤ cookies[i] ≤ 10⁵`; `2 ≤ k ≤ n`.

In [1]:
from typing import List
from itertools import product
from math import inf
import heapq, random, time

examples = [([8, 15, 10, 20, 8], 2, 31), ([6, 1, 3, 2, 2, 4, 1, 2], 3, 7)]
for c, k, exp in examples:
    print(f"cookies={c} k={k} -> kỳ vọng {exp}; tổng = {sum(c)}, trung bình = {sum(c) / k:.2f}, túi lớn nhất = {max(c)}")

cookies=[8, 15, 10, 20, 8] k=2 -> kỳ vọng 31; tổng = 61, trung bình = 30.50, túi lớn nhất = 20
cookies=[6, 1, 3, 2, 2, 4, 1, 2] k=3 -> kỳ vọng 7; tổng = 21, trung bình = 7.00, túi lớn nhất = 6


## 3. Kiến thức nền

### 3.1 Hai chặn dưới hiển nhiên

- Phải có em nhận ít nhất **trung bình**: `ans ≥ ⌈Σ / k⌉`.
- Em nhận túi lớn nhất có ít nhất `max(cookies)`: `ans ≥ max(cookies)`.

Chặn dưới `L = max(⌈Σ/k⌉, max)` **không phải lúc nào** cũng đạt được: hai ví dụ của đề đạt đúng `L`, nhưng `[6, 5, 5, 4, 4]` với `k = 2` có `L = 12` mà không tập con nào có tổng 12 (cell dưới).

In [2]:
for c, k, exp in examples:
    L = max(-(-sum(c) // k), max(c))
    print(f"{c}, k={k}: chặn dưới {L}, đáp án {exp}")
c = [6, 5, 5, 4, 4]
sums = sorted({sum(c[i] for i in range(5) if m >> i & 1) for m in range(32)})
print(f"{c}, k=2: chặn dưới {max(-(-sum(c) // 2), max(c))}, nhưng các tổng tập con là {sums} — không có 12 => không chia đều được")

[8, 15, 10, 20, 8], k=2: chặn dưới 31, đáp án 31
[6, 1, 3, 2, 2, 4, 1, 2], k=3: chặn dưới 7, đáp án 7
[6, 5, 5, 4, 4], k=2: chặn dưới 12, nhưng các tổng tập con là [0, 4, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 18, 19, 20, 24] — không có 12 => không chia đều được


### 3.2 Không gian tìm kiếm: `kⁿ` cách gán

Mỗi túi chọn một trong `k` em ⇒ `kⁿ ≤ 8⁸ ≈ 1,7·10⁷` — vét cạn thô trong Python mất hàng chục giây. Cần cắt tỉa hoặc DP.

### 3.3 Duyệt tập con của một mặt nạ

`sub = mask; while sub: …; sub = (sub − 1) & mask` liệt kê mọi tập con khác rỗng của `mask` theo thứ tự giảm. Tổng số cặp (mask, sub ⊆ mask) trên mọi `mask` là `3ⁿ` (mỗi phần tử: không thuộc mask / thuộc mask nhưng không thuộc sub / thuộc sub).

In [3]:
mask = 0b1011; subs = []; sub = mask
while sub:
    subs.append(bin(sub)); sub = (sub - 1) & mask
print(subs)
print("tổng số cặp (mask, sub) với n=8:", sum(1 << bin(m).count("1") for m in range(1 << 8)), "= 3^8 =", 3 ** 8)

['0b1011', '0b1010', '0b1001', '0b1000', '0b11', '0b10', '0b1']
tổng số cặp (mask, sub) với n=8: 6561 = 3^8 = 6561


### 3.4 Tổng của mọi tập con trong O(2ⁿ)

`ssum[mask] = ssum[mask & (mask − 1)] + cookies[bit thấp nhất của mask]`.

In [4]:
c = [8, 15, 10]
ssum = [0] * (1 << len(c))
for mask in range(1, 1 << len(c)):
    low = (mask & -mask).bit_length() - 1
    ssum[mask] = ssum[mask & (mask - 1)] + c[low]
print([(bin(m), ssum[m]) for m in range(1 << len(c))])

[('0b0', 0), ('0b1', 8), ('0b10', 15), ('0b11', 23), ('0b100', 10), ('0b101', 18), ('0b110', 25), ('0b111', 33)]


## 4. Ý tưởng nền tảng / trực giác

Đây là bài toán "chia `n` công việc cho `k` máy để thời gian hoàn thành (máy bận nhất) nhỏ nhất". Không có công thức đóng; với `n ≤ 8` ta vét cạn **thông minh**:

**Quay lui:** lần lượt đặt túi `i` cho một em `j`; giữ `load[j]`. Khi đặt xong mọi túi, cập nhật `ans = min(ans, max(load))`. Cắt tỉa:
1. **Đáp án hiện có:** nếu `load[j] + cookies[i] ≥ ans` thì đặt túi này cho `j` không thể cho kết quả **tốt hơn** ⇒ bỏ qua.
2. **Đối xứng:** hai em có cùng `load` hiện tại là hoán đổi được ⇒ chỉ thử một em (đặc biệt: chỉ thử **một** em đang có 0 bánh).
3. **Túi lớn trước:** sắp giảm dần ⇒ `load` tăng nhanh, cắt tỉa sớm hơn, và tìm được đáp án tốt sớm.

Tính tay ví dụ 1 (sắp giảm dần `[20, 15, 10, 8, 8]`, `k = 2`): túi 20 cho em A (em B đối xứng — bỏ). Túi 15: A (35) hoặc B (15). Nhánh B: `[20 | 15]` → túi 10: A = 30 hoặc B = 25 → … nhánh `A = 20 + 10 = 30`, `B = 15 + 8 + 8 = 31` cho `ans = 31`; mọi nhánh khác hoặc ra ≥ 31 hoặc bị cắt.

In [5]:
def all_assignments_best(cookies, k):
    best = inf; arg = None
    for assign in product(range(k), repeat=len(cookies)):
        load = [0] * k
        for c, j in zip(cookies, assign): load[j] += c
        if max(load) < best: best, arg = max(load), assign
    return best, arg

best, arg = all_assignments_best(examples[0][0], 2)
print("tối ưu:", best, "| cách chia:", [[c for c, j in zip(examples[0][0], arg) if j == t] for t in range(2)])

tối ưu: 31 | cách chia: [[8, 15, 8], [10, 20]]


## 5. Cách 1 — Brute Force / baseline

### 5.1 Duyệt mọi `kⁿ` cách gán — O(kⁿ · n)

In [6]:
def brute(cookies, k):
    return all_assignments_best(cookies, k)[0]

for c, k, exp in examples:
    t0 = time.perf_counter(); r = brute(c, k); dt = time.perf_counter() - t0
    print(r, "OK" if r == exp else "SAI", f"({k}^{len(c)} = {k ** len(c)} cách, {dt*1000:.0f} ms)")

31 OK (2^5 = 32 cách, 0 ms)
7 OK (3^8 = 6561 cách, 3 ms)


Với `n = k = 8`: `8⁸ ≈ 1,7·10⁷` cách, mỗi cách O(n) ⇒ ~1–2 phút trong Python. Quá chậm.

### 5.2 Baseline 2 — Quay lui không cắt tỉa (chỉ tránh tạo tuple)

In [7]:
def backtrack_plain(cookies, k):
    load = [0] * k; best = [inf]; n = len(cookies)
    def dfs(i):
        if i == n:
            best[0] = min(best[0], max(load)); return
        for j in range(k):
            load[j] += cookies[i]; dfs(i + 1); load[j] -= cookies[i]
    dfs(0)
    return best[0]

for c, k, exp in examples:
    print(backtrack_plain(c, k), "OK" if backtrack_plain(c, k) == exp else "SAI")

31 OK
7 OK


Vẫn duyệt đủ `kⁿ` lá. **Nút thắt:** (a) rất nhiều nhánh đã vượt đáp án tốt nhất mà vẫn đi tiếp; (b) mỗi cách chia được đếm `k!` lần do hoán vị tên các em. Cắt tỉa hai điều này giảm số nút hàng nghìn lần.

## 6. Ý tưởng thuật toán / bước đột phá

### 6.1 Quay lui có cắt tỉa (cách doocs)

```
sort cookies giảm dần; ans = inf; load = [0]*k
dfs(i):
    if i == n: ans = max(load); return          # đã cắt tỉa nên max(load) < ans
    for j in 0..k-1:
        if load[j] + c[i] >= ans: continue      # không thể tốt hơn
        if j > 0 and load[j] == load[j-1]: continue   # đối xứng (đơn giản hóa)
        load[j] += c[i]; dfs(i+1); load[j] -= c[i]
```

*Ghi chú về phá đối xứng:* điều kiện chuẩn là "đã thử một em có cùng `load` này rồi" (dùng tập `seen`). Điều kiện `load[j] == load[j−1]` của doocs chỉ so với em **liền trước** — vẫn đúng (không bỏ sót) vì chỉ bỏ qua khi tồn tại một em khác có cùng `load` đã được thử; nó có thể bỏ sót một số cơ hội cắt tỉa nhưng không làm sai.

### 6.2 DP bitmask

`f[t][mask]` = độ bất công nhỏ nhất khi chia **đúng** các túi trong `mask` cho `t` em (cho phép em nhận rỗng):

`f[1][mask] = ssum[mask]`, `f[t][mask] = min_{sub ⊆ mask} max(f[t−1][mask \ sub], ssum[sub])`.

Đáp án `f[k][full]`. Độ phức tạp O(k · 3ⁿ) = `8 · 6561 ≈ 5·10⁴` — rất nhỏ, **không phụ thuộc** dữ liệu.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quay lui có cắt tỉa (doocs)

In [8]:
def backtrack_pruned(cookies, k):
    cs = sorted(cookies, reverse=True); n = len(cs); load = [0] * k; ans = [inf]
    def dfs(i):
        if i == n:
            ans[0] = max(load); return
        for j in range(k):
            if load[j] + cs[i] >= ans[0] or (j and load[j] == load[j - 1]):
                continue
            load[j] += cs[i]; dfs(i + 1); load[j] -= cs[i]
    dfs(0)
    return ans[0]

for c, k, exp in examples:
    print(backtrack_pruned(c, k), "OK" if backtrack_pruned(c, k) == exp else "SAI")

31 OK
7 OK


Vì sao gán `ans = max(load)` (không cần `min`) ở lá? Mọi lần đặt túi đều thỏa `load[j] + c < ans`, nên ở lá `max(load) < ans` chắc chắn.

### 7.2 Cách 3 — Quay lui với tập `seen` (phá đối xứng đầy đủ)

In [9]:
def backtrack_seen(cookies, k):
    cs = sorted(cookies, reverse=True); n = len(cs); load = [0] * k; ans = [inf]
    def dfs(i):
        if i == n:
            ans[0] = max(load); return
        seen = set()
        for j in range(k):
            if load[j] in seen or load[j] + cs[i] >= ans[0]: continue
            seen.add(load[j])
            load[j] += cs[i]; dfs(i + 1); load[j] -= cs[i]
    dfs(0)
    return ans[0]

for c, k, exp in examples:
    print(backtrack_seen(c, k), "OK" if backtrack_seen(c, k) == exp else "SAI")

31 OK
7 OK


### 7.3 Cách 4 — DP bitmask O(k · 3ⁿ)

In [10]:
def bitmask_dp(cookies, k):
    n = len(cookies); full = (1 << n) - 1
    ssum = [0] * (1 << n)
    for mask in range(1, 1 << n):
        low = (mask & -mask).bit_length() - 1
        ssum[mask] = ssum[mask & (mask - 1)] + cookies[low]
    f = ssum[:]                                   # t = 1
    for _ in range(2, k + 1):
        g = [0] * (1 << n)
        for mask in range(1, 1 << n):
            best = f[mask]                        # em mới nhận rỗng
            sub = mask
            while sub:
                v = f[mask ^ sub]
                if ssum[sub] > v: v = ssum[sub]
                if v < best: best = v
                sub = (sub - 1) & mask
            g[mask] = best
        f = g
    return f[full]

for c, k, exp in examples:
    print(bitmask_dp(c, k), "OK" if bitmask_dp(c, k) == exp else "SAI")

31 OK
7 OK


### 7.4 Cách 5 — Tìm nhị phân đáp án + kiểm tra khả thi (quay lui "có chia được với ngưỡng `T`?")

`feasible(T)` đơn điệu: nếu chia được với ngưỡng `T` thì cũng chia được với mọi `T' ≥ T`. Tìm `T` nhỏ nhất trong `[max(cookies), Σ]`.

In [11]:
def binary_search_answer(cookies, k):
    cs = sorted(cookies, reverse=True); n = len(cs)
    def feasible(T):
        load = [0] * k
        def dfs(i):
            if i == n: return True
            seen = set()
            for j in range(k):
                if load[j] + cs[i] > T or load[j] in seen: continue
                seen.add(load[j])
                load[j] += cs[i]
                if dfs(i + 1): return True
                load[j] -= cs[i]
            return False
        return dfs(0)
    lo, hi = max(cs), sum(cs)
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid): hi = mid
        else: lo = mid + 1
    return lo

for c, k, exp in examples:
    print(binary_search_answer(c, k), "OK" if binary_search_answer(c, k) == exp else "SAI")

31 OK
7 OK


## 8. TRACE từng bước

### 8.1 Đếm số nút của từng cách quay lui trên ví dụ 2

In [12]:
def count_nodes(cookies, k, prune_ans, prune_sym, sort_desc):
    cs = sorted(cookies, reverse=True) if sort_desc else list(cookies)
    n = len(cs); load = [0] * k; ans = [inf]; nodes = [0]
    def dfs(i):
        nodes[0] += 1
        if i == n:
            ans[0] = min(ans[0], max(load)); return
        for j in range(k):
            if prune_ans and load[j] + cs[i] >= ans[0]: continue
            if prune_sym and j and load[j] == load[j - 1]: continue
            load[j] += cs[i]; dfs(i + 1); load[j] -= cs[i]
    dfs(0)
    return ans[0], nodes[0]

c, k, _ = examples[1]
for pa, ps, sd in [(False, False, False), (True, False, False), (False, True, False), (True, True, False), (True, True, True)]:
    r, nd = count_nodes(c, k, pa, ps, sd)
    print(f"cắt theo ans={pa!s:<5} đối xứng={ps!s:<5} sắp giảm={sd!s:<5} -> đáp án {r}, số nút {nd}")

cắt theo ans=False đối xứng=False sắp giảm=False -> đáp án 7, số nút 9841
cắt theo ans=True  đối xứng=False sắp giảm=False -> đáp án 7, số nút 271
cắt theo ans=False đối xứng=True  sắp giảm=False -> đáp án 7, số nút 1359
cắt theo ans=True  đối xứng=True  sắp giảm=False -> đáp án 7, số nút 117
cắt theo ans=True  đối xứng=True  sắp giảm=True  -> đáp án 7, số nút 214


Mỗi kỹ thuật cắt tỉa giảm số nút; kết hợp cả ba giảm từ hàng nghìn xuống vài trăm (hoặc ít hơn). Lưu ý: khi **không** cắt theo `ans`, phải dùng `min` ở lá.

### 8.2 Trace DP bitmask trên ví dụ nhỏ `[3, 2, 2]`, `k = 2`

In [13]:
c = [3, 2, 2]; n = 3; k = 2
ssum = [sum(c[i] for i in range(n) if m >> i & 1) for m in range(1 << n)]
f1 = ssum[:]
f2 = []
for mask in range(1 << n):
    best = f1[mask]; arg = 0; sub = mask
    while sub:
        v = max(f1[mask ^ sub], ssum[sub])
        if v < best: best, arg = v, sub
        sub = (sub - 1) & mask
    f2.append(best)
    print(f"mask={mask:03b} túi={[c[i] for i in range(n) if mask >> i & 1]!s:<10} f1={f1[mask]:<2} f2={best:<2} (em thứ hai nhận {[c[i] for i in range(n) if arg >> i & 1]})")
print("đáp án:", f2[-1])

mask=000 túi=[]         f1=0  f2=0  (em thứ hai nhận [])
mask=001 túi=[3]        f1=3  f2=3  (em thứ hai nhận [])
mask=010 túi=[2]        f1=2  f2=2  (em thứ hai nhận [])
mask=011 túi=[3, 2]     f1=5  f2=3  (em thứ hai nhận [2])
mask=100 túi=[2]        f1=2  f2=2  (em thứ hai nhận [])
mask=101 túi=[3, 2]     f1=5  f2=3  (em thứ hai nhận [2])
mask=110 túi=[2, 2]     f1=4  f2=2  (em thứ hai nhận [2])
mask=111 túi=[3, 2, 2]  f1=7  f2=4  (em thứ hai nhận [2, 2])
đáp án: 4


## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def distributeCookies(self, cookies: List[int], k: int) -> int:
        n = len(cookies)
        full = (1 << n) - 1
        ssum = [0] * (1 << n)                     # tổng bánh của mỗi tập túi
        for mask in range(1, 1 << n):
            low = (mask & -mask).bit_length() - 1
            ssum[mask] = ssum[mask & (mask - 1)] + cookies[low]
        f = ssum[:]                               # f[mask]: bất công nhỏ nhất khi chia mask cho t em (t = 1)
        for _ in range(k - 1):
            g = f[:]                              # em mới có thể nhận rỗng
            for mask in range(1, 1 << n):
                sub = mask
                while sub:                        # sub: phần của em mới
                    v = max(f[mask ^ sub], ssum[sub])
                    if v < g[mask]:
                        g[mask] = v
                    sub = (sub - 1) & mask
            f = g
        return f[full]

In [15]:
sol = Solution()
for c, k, exp in examples:
    got = sol.distributeCookies(c, k)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

31 | kỳ vọng 31 | OK
7 | kỳ vọng 7 | OK


Bản nộp chọn DP bitmask vì thời gian **cố định** (không phụ thuộc dữ liệu xấu cho cắt tỉa) và dễ chứng minh. Quay lui cắt tỉa cũng qua dễ dàng với `n ≤ 8`.

## 10. Vì sao thuật toán đúng

**DP bitmask.** Định nghĩa `F(t, S)` = min trên mọi cách chia tập túi `S` cho `t` em (em có thể nhận rỗng) của tổng lớn nhất. Mọi cách chia `S` cho `t` em gồm: phần `T ⊆ S` của em thứ `t` và một cách chia `S \ T` cho `t − 1` em. Bất công của cách chia đó là `max(bất công phần còn lại, ssum[T])`. Với `T` cố định, phần còn lại tốt nhất là `F(t−1, S \ T)` và `max(·, ssum[T])` không giảm theo đối số đầu ⇒

`F(t, S) = min_{T ⊆ S} max(F(t−1, S \ T), ssum[T])`, `F(1, S) = ssum[S]`.

`T = ∅` cho `max(F(t−1, S), 0) = F(t−1, S)` — chính là khởi tạo `g = f[:]`; vòng `while sub` duyệt các `T ≠ ∅`. Đáp án `F(k, full)`. Cho phép em nhận rỗng không làm thay đổi đáp án (đề không cấm em nhận 0 túi; và vì `k ≤ n`, cũng không cần ép mỗi em ≥ 1 túi — một em rỗng chỉ có thể làm bất công lớn hơn hoặc bằng).

**Quay lui cắt tỉa.** (a) Cắt `load[j] + c ≥ ans`: mọi lá trong nhánh đó có `max(load) ≥ load[j] + c ≥ ans` — không cải thiện. (b) Phá đối xứng: nếu `load[j] = load[j']` thì hoán đổi tên hai em biến mọi cách chia trong nhánh "đặt túi cho `j`" thành một cách chia trong nhánh "đặt cho `j'`" với cùng bất công ⇒ chỉ cần thử một nhánh. (c) Sắp xếp không đổi tập cách chia. ⇒ Kết quả là tối ưu. ∎

**Tìm nhị phân.** `feasible(T)` đơn điệu; `feasible(Σ)` đúng (một em nhận hết); tìm nhị phân trả về `T` nhỏ nhất khả thi = đáp án.

## 11. Phân tích Time / Space Complexity

`n ≤ 8`, `k ≤ n`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi `kⁿ` cách gán | O(kⁿ · n) ≈ 1,3·10⁸ | O(n) |
| 1b. Quay lui không cắt | O(kⁿ) | O(n + k) |
| 2. Quay lui cắt tỉa (doocs) | O(kⁿ) worst, thực tế nhỏ | O(n + k) |
| 3. Quay lui + `seen` | như trên, cắt mạnh hơn | O(n·k) |
| 4. DP bitmask | O(k · 3ⁿ) ≈ 5·10⁴ | O(2ⁿ) |
| 5. Tìm nhị phân + khả thi | O(log Σ · kⁿ) worst | O(n + k) |

In [16]:
rng = random.Random(2305)
cases = [([rng.randint(1, 10**5) for _ in range(8)], k) for k in (2, 3, 5, 8)] + [([10**5] * 8, 3), ([1, 1, 1, 1, 1, 1, 1, 100000], 4)]
for c, k in cases:
    res = []
    line = f"k={k} "
    for name, f in [("pruned", backtrack_pruned), ("seen", backtrack_seen), ("bitmask", bitmask_dp),
                    ("bsearch", binary_search_answer), ("Solution", Solution().distributeCookies)]:
        t0 = time.perf_counter(); res.append(f(c, k)); dt = time.perf_counter() - t0
        line += f"{name}={dt*1000:6.1f}ms "
    assert len(set(res)) == 1
    print(line, "->", res[0])

k=2 pruned=   0.0ms seen=   0.0ms bitmask=   0.5ms bsearch=   0.1ms Solution=   0.5ms  -> 171773
k=3 pruned=   0.1ms seen=   0.1ms bitmask=   0.8ms bsearch=   0.1ms Solution=   1.0ms  -> 133774
k=5 pruned=   0.4ms seen=   0.4ms bitmask=   1.6ms bsearch=   0.1ms Solution=   1.8ms  -> 90356
k=8 pruned=   0.4ms seen=   0.4ms bitmask=   2.6ms bsearch=   0.1ms Solution=   3.0ms  -> 83505
k=3 pruned=   0.0ms seen=   0.0ms bitmask=   0.7ms bsearch=   0.1ms Solution=   0.8ms  -> 300000
k=4 pruned=   0.1ms seen=   0.1ms bitmask=   1.2ms bsearch=   0.0ms Solution=   1.2ms  -> 100000


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
ALL = {"brute": brute, "backtrack_plain": backtrack_plain, "backtrack_pruned": backtrack_pruned,
       "backtrack_seen": backtrack_seen, "bitmask_dp": bitmask_dp, "binary_search_answer": binary_search_answer,
       "Solution": Solution().distributeCookies}
edge = examples + [
    ([1, 1], 2, 1),
    ([5, 7], 2, 7),                                  # k = n: mỗi em một túi => max
    ([4, 4, 4, 4], 4, 4),
    ([3, 3, 2, 2, 2], 2, 6),                         # chia đúng trung bình: 3+3 | 2+2+2
    ([6, 5, 5, 4, 4], 2, 13),                        # trung bình 12 nhưng không có tập con tổng 12
    ([100000, 1, 1, 1], 2, 100000),                  # túi lớn quyết định
    ([10, 10, 10, 1], 3, 11),
]
for c, k, exp in edge:
    ref = brute(c, k)
    for name, f in ALL.items():
        assert f(list(c), k) == ref, (name, c, k)
    print(f"cookies={c!s:<28} k={k} -> {ref} | ghi trong bảng: {exp} | khớp: {ref == exp}")

cookies=[8, 15, 10, 20, 8]           k=2 -> 31 | ghi trong bảng: 31 | khớp: True
cookies=[6, 1, 3, 2, 2, 4, 1, 2]     k=3 -> 7 | ghi trong bảng: 7 | khớp: True
cookies=[1, 1]                       k=2 -> 1 | ghi trong bảng: 1 | khớp: True
cookies=[5, 7]                       k=2 -> 7 | ghi trong bảng: 7 | khớp: True
cookies=[4, 4, 4, 4]                 k=4 -> 4 | ghi trong bảng: 4 | khớp: True
cookies=[3, 3, 2, 2, 2]              k=2 -> 6 | ghi trong bảng: 6 | khớp: True
cookies=[6, 5, 5, 4, 4]              k=2 -> 13 | ghi trong bảng: 13 | khớp: True
cookies=[100000, 1, 1, 1]            k=2 -> 100000 | ghi trong bảng: 100000 | khớp: True
cookies=[10, 10, 10, 1]              k=3 -> 11 | ghi trong bảng: 11 | khớp: True


Case `[6, 5, 5, 4, 4]`, `k = 2`: tổng 24, trung bình 12. Các tập con chứa túi 6 có tổng `6, 10, 11, 14, 15, 16, 19, 20, 24` (và phần bù của chúng có tổng `18, 14, 13, 10, 9, 8, 5, 4, 0`) — không có cách chia 12–12. Cách tốt nhất là `{6, 5} = 11 | {5, 4, 4} = 13` ⇒ **13**. Đây là ví dụ điển hình cho thấy không được tin vào chặn dưới trung bình.

### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [18]:
rng = random.Random(42)
for _ in range(300):
    n = rng.randint(2, 6); k = rng.randint(2, n)
    c = [rng.randint(1, rng.choice([5, 20, 10**5])) for _ in range(n)]
    ref = brute(c, k)
    for name, f in ALL.items():
        assert f(list(c), k) == ref, (name, c, k)
print("300 bộ ngẫu nhiên OK")

300 bộ ngẫu nhiên OK


### 12.3 Tính chất: chặn dưới `max(⌈Σ/k⌉, max)`; không tăng khi `k` tăng; `k = n` ⇒ `max(cookies)`; hoán vị không đổi đáp án

In [19]:
rng = random.Random(7)
for _ in range(300):
    n = rng.randint(2, 8); c = [rng.randint(1, 50) for _ in range(n)]
    vals = [Solution().distributeCookies(c, k) for k in range(2, n + 1)]
    for k, v in zip(range(2, n + 1), vals):
        assert v >= max(-(-sum(c) // k), max(c))
    assert all(a >= b for a, b in zip(vals, vals[1:])) and vals[-1] == max(c)
    d = c[:]; rng.shuffle(d)
    assert Solution().distributeCookies(d, 2) == vals[0]
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam LPT: sắp giảm dần, mỗi túi cho em đang ít bánh nhất

In [20]:
def wrong_lpt(cookies, k):
    h = [0] * k
    for c in sorted(cookies, reverse=True):
        heapq.heappush(h, heapq.heappop(h) + c)            # SAI: heuristic, không tối ưu
    return max(h)

for c, k, _ in examples:
    print("bản sai :", wrong_lpt(c, k), "| bản đúng:", Solution().distributeCookies(c, k))
c, k = [3, 3, 2, 2, 2], 2
print("bản sai :", wrong_lpt(c, k), "| bản đúng:", Solution().distributeCookies(c, k), "cho", c, "k =", k)

bản sai : 33 | bản đúng: 31
bản sai : 7 | bản đúng: 7
bản sai : 7 | bản đúng: 6 cho [3, 3, 2, 2, 2] k = 2


**Vì sao sai:** ví dụ 1 đã sai: LPT đặt `20 | 15`, rồi `10` cho em 15 (`20 | 25`), `8` cho em 20 (`28 | 25`), `8` cho em 25 ⇒ `33 > 31`. Ví dụ 2 **trùng hợp** đúng. Với `[3, 3, 2, 2, 2]`, `k = 2`: LPT cho `3 | 3` rồi `2` cho một em (`5 | 3`), `2` cho em kia (`5 | 5`), `2` cuối ⇒ `7`; tối ưu là `3 + 3 = 6 | 2 + 2 + 2 = 6`. LPT chỉ đảm bảo xấp xỉ `4/3 − 1/(3k)` lần tối ưu.

### Lỗi 2 — Trả về chặn dưới `max(⌈Σ/k⌉, max(cookies))`

In [21]:
def wrong_lower_bound(cookies, k):
    return max(-(-sum(cookies) // k), max(cookies))       # SAI: chặn dưới, không phải đáp án

for c, k in [examples[0][:2], examples[1][:2], ([6, 5, 5, 4, 4], 2)]:
    print("bản sai :", wrong_lower_bound(c, k), "| bản đúng:", Solution().distributeCookies(c, k))

bản sai : 31 | bản đúng: 31
bản sai : 7 | bản đúng: 7
bản sai : 12 | bản đúng: 13


**Vì sao sai:** hai ví dụ đề **trùng hợp** đạt đúng chặn dưới; `[6, 5, 5, 4, 4]` thì không (không có tập con tổng 12).

### Lỗi 3 — Cắt tỉa bằng `break` thay vì `continue`

In [22]:
def wrong_break(cookies, k):
    cs = sorted(cookies, reverse=True); n = len(cs); load = [0] * k; ans = [inf]
    def dfs(i):
        if i == n:
            ans[0] = max(load); return
        for j in range(k):
            if load[j] + cs[i] >= ans[0]: break          # SAI: các em sau có thể ít bánh hơn em j
            if j and load[j] == load[j - 1]: continue
            load[j] += cs[i]; dfs(i + 1); load[j] -= cs[i]
    dfs(0)
    return ans[0]

found = None; rng = random.Random(11)
for _ in range(5000):
    n = rng.randint(3, 7); k = rng.randint(2, min(4, n)); c = [rng.randint(1, 12) for _ in range(n)]
    if wrong_break(c, k) != Solution().distributeCookies(c, k): found = (c, k); break
c, k = found
print("phản ví dụ:", c, "k =", k, "| bản sai :", wrong_break(c, k), "| bản đúng:", Solution().distributeCookies(c, k))
print("ví dụ đề:", [wrong_break(c, k) for c, k, _ in examples])

phản ví dụ: [8, 8, 9, 10, 4, 3] k = 4 | bản sai : 13 | bản đúng: 12
ví dụ đề: [31, 7]


**Vì sao sai:** `load` **không** được sắp theo `j`; em `j` quá tải không có nghĩa em `j + 1` cũng quá tải. **Sửa:** `continue`.

### Lỗi 4 — Phá đối xứng sai: bỏ qua mọi em có `load` **bằng một em bất kỳ** kể cả khi em đó chưa được thử

In [23]:
def wrong_symmetry(cookies, k):
    cs = sorted(cookies, reverse=True); n = len(cs); load = [0] * k; ans = [inf]
    def dfs(i):
        if i == n:
            ans[0] = min(ans[0], max(load)); return
        for j in range(k):
            if load[j] + cs[i] >= ans[0]: continue
            if any(load[j] == load[t] for t in range(k) if t != j): continue   # SAI: có thể bỏ cả hai
            load[j] += cs[i]; dfs(i + 1); load[j] -= cs[i]
    dfs(0)
    return ans[0]

for c, k, _ in examples:
    print("bản sai :", wrong_symmetry(c, k), "| bản đúng:", Solution().distributeCookies(c, k))

bản sai : inf | bản đúng: 31
bản sai : inf | bản đúng: 7


**Vì sao sai:** ngay ở túi đầu tiên mọi em đều có `load = 0` ⇒ mọi em đều bị bỏ ⇒ không có lá nào ⇒ `inf`. Phá đối xứng phải giữ lại **một** đại diện cho mỗi giá trị `load` (em đầu tiên, hoặc dùng `seen`).

### Lỗi 5 — DP bitmask dùng `min` thay vì `max` khi ghép phần của em mới

In [24]:
def wrong_dp_min(cookies, k):
    n = len(cookies); ssum = [sum(cookies[i] for i in range(n) if m >> i & 1) for m in range(1 << n)]
    f = ssum[:]
    for _ in range(k - 1):
        g = f[:]
        for mask in range(1, 1 << n):
            sub = mask
            while sub:
                g[mask] = min(g[mask], min(f[mask ^ sub], ssum[sub]))   # SAI: bất công là MAX của các em
                sub = (sub - 1) & mask
        f = g
    return f[-1]

for c, k, _ in examples:
    print("bản sai :", wrong_dp_min(c, k), "| bản đúng:", Solution().distributeCookies(c, k))

bản sai : 0 | bản đúng: 31
bản sai : 0 | bản đúng: 7


**Sửa:** `max(f[mask ^ sub], ssum[sub])` — bất công của cách chia là em nhận **nhiều nhất**; ta tối thiểu hóa đại lượng đó.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n ≤ 8`–`15` và "chia tập thành k nhóm tối ưu" ⇒ **quay lui cắt tỉa** hoặc **DP bitmask** (`O(k·3ⁿ)` hoặc `O(2ⁿ·n)`).
- Quay lui: **sắp giảm dần**, cắt theo đáp án tốt nhất, **phá đối xứng** (em có cùng tải / em rỗng).
- "Tối thiểu hóa giá trị lớn nhất" ⇒ cũng nghĩ tới **tìm nhị phân đáp án** + kiểm tra khả thi.
- Tham lam LPT chỉ là xấp xỉ.
- Bài cùng họ: 698 Partition to K Equal Sum Subsets, 1723 Find Minimum Time to Finish All Jobs, 473 Matchsticks to Square, 1986 Minimum Number of Work Sessions to Finish the Tasks, 2323 Find Minimum Time to Finish All Jobs II (khác: ghép cặp), 410 Split Array Largest Sum (bản đoạn liên tiếp).

## 15. Biến thể của bài

### 15.1 Biến thể A — Mỗi em phải nhận **ít nhất một** túi

Với bài gốc điều này không đổi đáp án khi `k ≤ n` (luôn tách được một túi từ em có ≥ 2 túi cho em rỗng mà không tăng max). Kiểm chứng:

In [25]:
def at_least_one_brute(cookies, k):
    best = inf
    for assign in product(range(k), repeat=len(cookies)):
        if len(set(assign)) < k: continue
        load = [0] * k
        for c, j in zip(cookies, assign): load[j] += c
        best = min(best, max(load))
    return best

rng = random.Random(151)
for _ in range(200):
    n = rng.randint(2, 6); k = rng.randint(2, n); c = [rng.randint(1, 20) for _ in range(n)]
    assert at_least_one_brute(c, k) == Solution().distributeCookies(c, k)
print("ràng buộc 'mỗi em >= 1 túi' không đổi đáp án")

ràng buộc 'mỗi em >= 1 túi' không đổi đáp án


### 15.2 Biến thể B — Tối đa hóa **phần nhỏ nhất** (max-min fairness)

In [26]:
def maximize_min(cookies, k):
    n = len(cookies); ssum = [sum(cookies[i] for i in range(n) if m >> i & 1) for m in range(1 << n)]
    f = ssum[:]                                    # f[mask]: phần nhỏ nhất lớn nhất khi chia mask cho t em
    for _ in range(k - 1):
        g = [-inf] * (1 << n)
        for mask in range(1 << n):
            sub = mask
            while True:
                g[mask] = max(g[mask], min(f[mask ^ sub], ssum[sub]))
                if sub == 0: break
                sub = (sub - 1) & mask
        f = g
    return f[-1]

def maximize_min_brute(cookies, k):
    best = -inf
    for assign in product(range(k), repeat=len(cookies)):
        load = [0] * k
        for c, j in zip(cookies, assign): load[j] += c
        best = max(best, min(load))
    return best

rng = random.Random(152)
for _ in range(200):
    n = rng.randint(2, 6); k = rng.randint(2, n); c = [rng.randint(1, 20) for _ in range(n)]
    assert maximize_min(c, k) == maximize_min_brute(c, k)
print(maximize_min(*examples[0][:2]), maximize_min(*examples[1][:2]))

30 7


### 15.3 Biến thể C — Trả về **cách chia** tối ưu (truy vết DP bitmask)

In [27]:
def best_distribution(cookies, k):
    n = len(cookies); full = (1 << n) - 1
    ssum = [sum(cookies[i] for i in range(n) if m >> i & 1) for m in range(1 << n)]
    layers = [ssum[:]]; choice = []
    for _ in range(k - 1):
        f = layers[-1]; g = f[:]; ch = [0] * (1 << n)
        for mask in range(1, 1 << n):
            sub = mask
            while sub:
                v = max(f[mask ^ sub], ssum[sub])
                if v < g[mask]: g[mask], ch[mask] = v, sub
                sub = (sub - 1) & mask
        layers.append(g); choice.append(ch)
    parts = []; mask = full
    for t in range(k - 2, -1, -1):
        sub = choice[t][mask]; parts.append([cookies[i] for i in range(n) if sub >> i & 1]); mask ^= sub
    parts.append([cookies[i] for i in range(n) if mask >> i & 1])
    return layers[-1][full], parts[::-1]

for c, k, exp in examples:
    val, parts = best_distribution(c, k)
    assert val == exp and max(map(sum, parts)) == val and sorted(x for p in parts for x in p) == sorted(c)
    print(val, parts)

31 [[10, 20], [8, 15, 8]]
7 [[6, 1], [3, 2, 2], [4, 1, 2]]


### 15.4 Biến thể D — Dãy túi phải chia thành **đoạn liên tiếp** (LC 410 Split Array Largest Sum)

Không còn là tập con tùy ý ⇒ DP đoạn hoặc tìm nhị phân + tham lam O(n log Σ), mở rộng được cho `n` lớn.

In [28]:
def split_contiguous(cookies, k):
    def parts_needed(T):
        cnt, cur = 1, 0
        for c in cookies:
            if cur + c > T: cnt += 1; cur = 0
            cur += c
        return cnt
    lo, hi = max(cookies), sum(cookies)
    while lo < hi:
        mid = (lo + hi) // 2
        if parts_needed(mid) <= k: hi = mid
        else: lo = mid + 1
    return lo

print(split_contiguous(*examples[0][:2]), "(đoạn liên tiếp) so với", examples[0][2], "(tập con tùy ý)")

33 (đoạn liên tiếp) so với 31 (tập con tùy ý)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi `kⁿ` cách | định nghĩa | O(kⁿ·n) | O(n) | kiểm chứng (n nhỏ) |
| 2. Quay lui cắt tỉa | doocs | nhỏ trong thực tế | O(n+k) | phổ biến khi phỏng vấn |
| 3. Quay lui + `seen` | phá đối xứng đầy đủ | nhỏ hơn nữa | O(n·k) | dữ liệu nhiều túi bằng nhau |
| 4. DP bitmask | tập con của tập con | O(k·3ⁿ) | O(2ⁿ) | **mặc định**: thời gian cố định |
| 5. Tìm nhị phân + khả thi | min-max | O(log Σ · …) | O(n+k) | khi kiểm tra khả thi dễ hơn tối ưu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `distributeCookies([1, 2, 3, 4], 2)` và `distributeCookies([9, 1, 1, 1], 3)`?

**Bài 2 (tính tay):** `cookies = [7, 6, 5, 4]`, `k = 2`. Liệt kê các cách chia gần trung bình và đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(cookies, k, T)` trả về `True` ⇔ chia được sao cho mọi em nhận ≤ `T`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(cookies, k)` cài **quay lui cắt tỉa** không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(cookies, k)` trả về **số cách chia** (theo nghĩa gán túi → em có đánh số) đạt độ bất công tối ưu.

In [29]:
def bt3_todo(cookies, k, T):
    # TODO: kiểm tra khả thi với ngưỡng T
    pass

def bt4_todo(cookies, k):
    # TODO: quay lui cắt tỉa
    pass

def bt5_todo(cookies, k):
    # TODO: số cách gán đạt tối ưu
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tổng 10, trung bình 5. Mức 2: `{1, 4} | {2, 3}` ⇒ 5. Mức 3: `[9, 1, 1, 1]`, `k = 3`: túi 9 quyết định ⇒ `9`.

**Bài 2.** Mức 1: tổng 22, trung bình 11. Mức 2: `{7, 4} = 11`, `{6, 5} = 11`. Mức 3: `11`.

**Bài 3.** Mức 1: quay lui đặt từng túi, bỏ em nếu vượt `T`. Mức 2: sắp giảm dần + `seen`. Mức 3: trả `True` khi đặt hết.

**Bài 4.** Mức 1: `load = [0] * k`, `ans = inf`. Mức 2: `continue` nếu `load[j] + c ≥ ans`. Mức 3: phá đối xứng theo em liền trước / `seen`.

**Bài 5.** Mức 1: tính đáp án tối ưu `best` trước. Mức 2: DP đếm: `cnt[t][mask]` = số cách chia `mask` cho `t` em (có thứ tự) với mọi em ≤ `best`. Mức 3: `cnt[t][mask] = Σ_{sub ⊆ mask, ssum[sub] ≤ best} cnt[t−1][mask \ sub]` (kể cả `sub = ∅`).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `5`, `9`. **Bài 2:** `11`.

In [30]:
assert Solution().distributeCookies([1, 2, 3, 4], 2) == 5
assert Solution().distributeCookies([9, 1, 1, 1], 3) == 9
assert Solution().distributeCookies([7, 6, 5, 4], 2) == 11

def bt3(cookies, k, T):
    cs = sorted(cookies, reverse=True); load = [0] * k
    def dfs(i):
        if i == len(cs): return True
        seen = set()
        for j in range(k):
            if load[j] + cs[i] > T or load[j] in seen: continue
            seen.add(load[j]); load[j] += cs[i]
            if dfs(i + 1): return True
            load[j] -= cs[i]
        return False
    return dfs(0)

def bt4(cookies, k):
    return backtrack_pruned(cookies, k)

def bt5(cookies, k):
    best = Solution().distributeCookies(cookies, k); n = len(cookies)
    ssum = [sum(cookies[i] for i in range(n) if m >> i & 1) for m in range(1 << n)]
    cnt = [1 if ssum[m] <= best else 0 for m in range(1 << n)]          # t = 1
    for _ in range(k - 1):
        new = [0] * (1 << n)
        for mask in range(1 << n):
            sub = mask
            while True:
                if ssum[sub] <= best: new[mask] += cnt[mask ^ sub]
                if sub == 0: break
                sub = (sub - 1) & mask
        cnt = new
    return cnt[-1]

def bt5_brute(cookies, k):
    best = brute(cookies, k); total = 0
    for assign in product(range(k), repeat=len(cookies)):
        load = [0] * k
        for c, j in zip(cookies, assign): load[j] += c
        total += max(load) == best
    return total

rng = random.Random(1994)
for _ in range(200):
    n = rng.randint(2, 6); k = rng.randint(2, n); c = [rng.randint(1, 15) for _ in range(n)]
    best = brute(c, k)
    assert bt3(c, k, best) and not bt3(c, k, best - 1)
    assert bt4(c, k) == best
    assert bt5(c, k) == bt5_brute(c, k)
print(bt5([1, 2, 3, 4], 2), bt5(*examples[0][:2]))
print("Đáp án bài tập: tất cả assert OK")

2 2
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# DP bitmask O(k * 3^n)
ssum[mask] = ssum[mask & (mask-1)] + c[lowbit(mask)]
f = ssum[:]                                   # 1 em
for _ in range(k - 1):
    g = f[:]                                  # em mới nhận rỗng
    for mask: for sub ⊆ mask (sub ≠ 0):
        g[mask] = min(g[mask], max(f[mask ^ sub], ssum[sub]))
    f = g
return f[full]
```

**Checklist:**
- [ ] Bất công là **max** tải; tối thiểu hóa nó (`min` ngoài, `max` trong).
- [ ] Quay lui: cắt tỉa bằng `continue` (không `break`), phá đối xứng giữ lại một đại diện.
- [ ] Không trả về chặn dưới trung bình; không dùng tham lam LPT.
- [ ] Duyệt tập con: `sub = (sub - 1) & mask`.